# 03a — Fixed-log: Deploy and Monitor

Deploy the complete model from `02a or 02b` (or the ready-made sample model in
`sample_data/models/`, if you haven't run it) and keep scoring new data with the
`amygda-risk` monitor, on the sample data split into 13 weekly files (Jan–Mar 2024).

See `03_deploy_and_monitor_guide.ipynb` for what each step does and why.

## Setup

In [ ]:
import glob, json, os, shutil, sys, webbrowser
from pathlib import Path

import pandas as pd

# Installing the SDK first time:
#   !pip install git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git
# Already installed, please update to the latest version:
#   !pip install --upgrade git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

import amygda_ops_risk_score
from amygda_ops_risk_score import OpsRiskClient, helpers
from amygda_ops_risk_score.monitor import Monitor, MonitorError

# The `amygda-risk` command. If this notebook's kernel can't find it on PATH (its
# environment isn't activated), the same command is run through this kernel's Python.
AMYGDA_RISK = shutil.which("amygda-risk") or f'"{sys.executable}" -m amygda_ops_risk_score.monitor.cli'

print("amygda_ops_risk_score version:", amygda_ops_risk_score.__version__)

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

API_KEY = os.environ.get("AMYGDA_SDK_API_KEY") or "paste_your_api_key_here"   # ← paste your key here

# The monitor and the `amygda-risk` command read the key from AMYGDA_API_KEY.
# It is never written to the monitor folder.
os.environ["AMYGDA_API_KEY"] = API_KEY

client = OpsRiskClient()
client.wait_until_ready()
print("API is ready.")

In [ ]:
# ── Paths — edit once ────────────────────────────────────────────────────────
# Yours from 02a or 02b if you ran it, otherwise the ready-made sample model.
MODEL_ZIPS = sorted(glob.glob("outputs/fixed_log/complete_model_*.zip"), key=os.path.getmtime)
MODEL      = MODEL_ZIPS[-1] if MODEL_ZIPS else "sample_data/models/fixed_log_complete_model.zip"
TRAINING   = "sample_data/fixed_log_training.csv"     # earlier logs, used as history
BATCHES    = Path("sample_data/batches/fixed_log")    # week_01.csv … week_13.csv
MON_DIR    = Path("outputs/fixed_log_monitor")
MON_NAME   = "example-fixed-log"

week = {int(p.stem[-2:]): p for p in sorted(BATCHES.glob("week_*.csv"))}
print("Model:", MODEL)

# Start clean: remove the folder and deployment from an earlier run of this example
for d in client.list_deployments(API_KEY):
    if d["name"] == MON_NAME:
        client.delete_deployment(API_KEY, d["deployment_id"])
shutil.rmtree(MON_DIR, ignore_errors=True)

## Deploy and create the monitor

Deploying keeps a small scoring bundle on Amygda's servers (no log rows, asset IDs or
timestamps) until you delete it or it goes unused for 90 days.

In [ ]:
!{AMYGDA_RISK} init {MON_DIR} --model {MODEL} --name {MON_NAME} \
    --history {TRAINING} --fleet-name "Baggage sortation stations"

Check the deployment scores exactly like the model's Step 12 run (expect `match: True`):

In [ ]:
helpers.check_deployment(client, API_KEY, Monitor(MON_DIR).deployment["deployment_id"], MODEL)

## Run the first weeks

`--input` takes one or more files. The period holding each run's newest log is marked
provisional until the next run completes it.

In [ ]:
!{AMYGDA_RISK} run {MON_DIR} --input {week[1]}

In [ ]:
for n in range(2, 8):
    !{AMYGDA_RISK} run {MON_DIR} --input {week[n]} --no-dashboard

## The rest through the inbox

With no `--input`, a run scores everything in `inbox/` and moves the files to
`inbox/processed/`. The fleet app (`04a_fleet_app_fixed_log.ipynb`) does this on a schedule.

In [ ]:
for n in range(8, 14):
    shutil.copy(week[n], MON_DIR / "inbox" / week[n].name)
!{AMYGDA_RISK} run {MON_DIR}

## Status, runs and scores

In [ ]:
!{AMYGDA_RISK} status {MON_DIR}

In [ ]:
mon = Monitor(MON_DIR)
cols = ["run_id", "trigger", "status", "rows_new", "periods_upserted", "unmatched_share",
        "provisional_periods", "warnings"]
pd.DataFrame(mon.runs()).reindex(columns=cols)

In [ ]:
scores = mon.scores
print(f"{len(scores):,} periods for {scores['asset_id'].nunique()} assets, "
      f"{scores['period_start'].min()} to {scores['period_start'].max()}")
(scores.groupby("asset_id")["operational_risk"].agg(["mean", "max"]).round(1)
       .sort_values("mean", ascending=False))

## Dashboard

In [ ]:
dash = (MON_DIR / "dashboard" / "index.html").resolve()
print(dash)
webbrowser.open(dash.as_uri())

## One-off scoring

Score a file with the same model without changing the monitor.

In [ ]:
!{AMYGDA_RISK} score {MON_DIR} --input {week[3]} --output outputs/{MON_NAME}_oneoff

## Clean up (optional)

Deletes the deployment and the monitor folder.

In [ ]:
DELETE = False   # ← set True to delete
if DELETE:
    client.delete_deployment(API_KEY, mon.deployment["deployment_id"])
    shutil.rmtree(MON_DIR, ignore_errors=True)
    shutil.rmtree(f"outputs/{MON_NAME}_oneoff", ignore_errors=True)